## 1. Import required Libraries

In [58]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

SEED = 376
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 2. Import data

In [59]:
curr_path = Path.cwd()
Output_Datasets_path = curr_path.parent.parent / 'Output_Datasets'

In [60]:
X_train_clean_path = Output_Datasets_path / 'X_train_clean.npy'
Y_train_clean_path = Output_Datasets_path / 'Y_train_clean.npy'
X_val_clean_path = Output_Datasets_path / 'X_val_clean.npy'
Y_val_clean_path = Output_Datasets_path / 'Y_val_clean.npy'

X_train_clean_short = Output_Datasets_path / 'X_train_clean_short.npy'
Y_train_clean_short = Output_Datasets_path / 'Y_train_clean_short.npy'
X_val_clean_short = Output_Datasets_path / 'X_val_clean_short.npy'
Y_val_clean_short = Output_Datasets_path / 'Y_val_clean_short.npy'

In [61]:
# print("Loading X_train_clean...")
# X_train_clean = np.load(X_train_clean_path, allow_pickle=True)
# print("Loading Y_train_clean...")
# Y_train_clean = np.load(Y_train_clean_path, allow_pickle=True)
# print("Successfully loaded X_train_clean and Y_train_clean")

In [62]:
print("Loading X_train_clean_short...")
X_train_clean_short = np.load(X_train_clean_short, allow_pickle=True)
print("Loading Y_train_clean_short...")
Y_train_clean_short = np.load(Y_train_clean_short, allow_pickle=True)
print("Successfully loaded X_train_clean_short and Y_train_clean_short")

Loading X_train_clean_short...
Loading Y_train_clean_short...
Successfully loaded X_train_clean_short and Y_train_clean_short


In [63]:
# print("Loading X_val_clean...")
# X_val_clean = np.load(X_val_clean_path, allow_pickle=True)
# print("Loading Y_val_clean...")
# Y_val_clean = np.load(Y_val_clean_path, allow_pickle=True)
# print("Successfully loaded X_val_clean and Y_val_clean")

In [64]:
print("Loading Y_val_clean_short...")
X_val_clean_short = np.load(X_val_clean_short, allow_pickle=True)
print("Loading Y_val_clean_short...")
Y_val_clean_short = np.load(Y_val_clean_short, allow_pickle=True)
print("Successfully loaded Y_val_clean_short and Y_val_clean_short")

Loading Y_val_clean_short...
Loading Y_val_clean_short...
Successfully loaded Y_val_clean_short and Y_val_clean_short


In [65]:
# print(f'Shape of X_train_clean: {X_train_clean.shape}')
# print(f'Shape of Y_train_clean: {Y_train_clean.shape}')

In [66]:
# print(f'Shape of X_val_clean: {X_val_clean.shape}')
# print(f'Shape of Y_val_clean: {Y_val_clean.shape}')

In [67]:
print(X_train_clean_short[0])
print(Y_train_clean_short[290])

[[-0.7290525  -0.3258207   0.38159752 ... -0.0825962  -0.13979171
  -0.27320004]
 [-0.71040192 -0.30146305  0.387582   ... -0.09935392 -0.12149178
  -0.25536681]
 [-0.73526937 -0.25883717  0.45341134 ... -0.08929929 -0.10319185
  -0.23753358]
 ...
 [ 0.43971742  0.00909697 -0.41433899 ...  0.08498095 -0.14711168
  -0.19830048]
 [ 0.54540406  0.03345462 -0.48615281 ...  0.81561734 -0.1654116
  -0.3409663 ]
 [ 0.14752494 -0.17967481 -0.32457172 ...  0.48381457 -0.12515176
  -0.41943251]]
[1 0 0 0 0]


In [68]:
X_train_clean_short[0].shape

(1000, 12)

## 3. PyTorch DataSet 

In [69]:
class EcgDataset(Dataset):
    def __init__(self, inputs, target):
        self.inputs = inputs
        self.target = target
    def __len__(self):
        return len(self.inputs)
    def __getitem__(self, idx):
        # Shape: (1000, 12) -> (12, 1000) for 1D convolution across time
        x = torch.tensor(self.inputs[idx], dtype=torch.float32).permute(1, 0)
        y = torch.tensor(self.target[idx], dtype=torch.float32)
        return x, y

In [70]:
print('Preparing dataset...')
train_dataset = EcgDataset(inputs=X_train_clean_short, target=Y_train_clean_short)
val_dataset = EcgDataset(inputs=X_val_clean_short, target=Y_val_clean_short)
print('Completed dataset preparation!')

Preparing dataset...
Completed dataset preparation!


## 4. PyTorch Dataloader

In [71]:
BATCH_SIZE = 32

In [72]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [73]:
print(len(train_dataloader))

63


In [74]:
for i, (x, _) in enumerate(train_dataloader):
    if i == 2:
        print(x[2].shape)
        break
    print(x[0])
    print(x[1])
    print(i)

tensor([[11.2508,  8.5403,  3.6663,  ..., -0.8720, -1.0337, -1.0150],
        [ 6.0011,  4.7040,  0.7581,  ..., -1.1540, -1.1601, -1.1783],
        [-4.9326, -3.6041, -2.7842,  ..., -0.2946, -0.1450, -0.1809],
        ...,
        [ 2.7159, -0.1965, -3.1560,  ..., -0.2468, -0.2904, -0.2401],
        [ 6.1224,  2.1184, -2.7274,  ..., -0.8974, -0.9633, -0.9011],
        [ 8.0264,  4.0710, -1.4145,  ..., -0.9295, -0.9723, -0.9580]])
tensor([[ 1.2914,  1.1982,  0.8438,  ...,  0.3340,  0.3402,  0.3651],
        [ 2.2865,  2.2196,  2.0369,  ...,  1.3122,  1.3183,  1.3305],
        [ 1.0100,  1.0219,  1.1895,  ...,  0.9741,  0.9681,  0.9561],
        ...,
        [ 3.4198,  3.3427,  3.0980,  ..., -1.7383, -1.7584, -1.7483],
        [ 2.6674,  2.5906,  2.3746,  ..., -1.3329, -1.3403, -1.3512],
        [ 2.2021,  2.1593,  1.9845,  ..., -1.1149, -1.1042, -1.0935]])
0
tensor([[-1.0088, -1.0026, -1.0275,  ..., -0.3001, -0.3312, -0.2628],
        [ 0.7094,  0.7398,  0.8555,  ..., -1.0992, -1.0992, 

In [75]:
for i, (x,_) in enumerate(train_dataloader):
    if i == 2:
        break
    print(x[i].shape)
    print(x[i][0].shape)
    print(x[i])
    

torch.Size([12, 1000])
torch.Size([1000])
tensor([[-0.3436, -0.4182, -0.3871,  ..., -0.4306, -0.6296,  0.1724],
        [ 0.3988,  0.3440,  0.3988,  ..., -0.1614, -0.2467,  0.5693],
        [ 0.7227,  0.7407,  0.7646,  ...,  0.2559,  0.3636,  0.3936],
        ...,
        [ 0.0615,  0.0280,  0.1353,  ..., -0.1798, -0.0089,  0.1587],
        [-0.0154, -0.0812,  0.0542,  ..., -0.1435, -0.1435, -0.2313],
        [-0.1662, -0.2732, -0.3231,  ..., -0.1555, -0.1912, -0.1912]])
torch.Size([12, 1000])
torch.Size([1000])
tensor([[-0.2877, -0.3374, -0.4058,  ...,  0.9557,  0.8500,  0.8438],
        [-0.7643, -0.7521, -0.7643,  ...,  1.4645,  1.3244,  1.4036],
        [-0.4742, -0.4143, -0.3605,  ...,  0.5192,  0.4773,  0.5671],
        ...,
        [ 0.1755,  0.1855,  0.2392,  ...,  0.1118,  0.0247, -0.1262],
        [-0.1764, -0.1069, -0.0080,  ...,  0.3104,  0.2152,  0.2079],
        [-0.3089, -0.2803, -0.2375,  ...,  0.5257,  0.4437,  0.3938]])


## 4. 2D CNN Model

In [76]:
#input shape of 2d CNN: [BATCH_SIZE, IN_CHANNELS, HEIGHT, WIDTH]

In [77]:
class ECG_CNN_1D(nn.Module):
    def __init__(self, in_channels=12, num_classes=5):
        super().__init__()
        
        # Block 1 (Local wave morphology: QRS, P, T waves)
        self.block1 = nn.Sequential(
            nn.Conv1d(in_channels=in_channels, out_channels=64, kernel_size=7, stride=1, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # 1000 -> 500
            nn.Dropout(0.2)
        )
        
        # Block 2 (Segment intervals: PR, ST)
        self.block2 = nn.Sequential(
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # 500 -> 250
            nn.Dropout(0.2)
        )
        
        # Block 3 (Beat-level patterns)
        self.block3 = nn.Sequential(
            nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # 250 -> 125
            nn.Dropout(0.2)
        )
        
        # Block 4 (Rhythm & multi-beat dependencies)
        self.block4 = nn.Sequential(
            nn.Conv1d(in_channels=256, out_channels=512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # 125 -> 62
            nn.Dropout(0.3)
        )
        
        # Block 5 (Global temporal context)
        self.block5 = nn.Sequential(
            nn.Conv1d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),      # Temporal pooling: (batch, 512, 62) -> (batch, 512, 1)
            nn.Dropout(0.3)
        )
        
        # Classification Head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)   # 5 raw logits for BCEWithLogitsLoss
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        return self.classifier(x)


In [78]:
model = ECG_CNN_1D(in_channels=12, num_classes=5).to(device)
print(model)

ECG_CNN_1D(
  (block1): Sequential(
    (0): Conv1d(12, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.2, inplace=False)
  )
  (block2): Sequential(
    (0): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.2, inplace=False)
  )
  (block3): Sequential(
    (0): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.2, inplace=False)
  )
  (block

In [ ]:
LEARNING_RATE = 1e-3
EPOCHS = 11

In [80]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

## Model Training Logic

In [81]:
def train_model(model, dataloader, criterion, optimizer, device, threshold=0.5):
    model.to(device)
    model.train()
    
    train_loss = 0.0
    total_samples = 0
    total_elements = 0
    correct_exact = 0
    correct_elements = 0
    
    # Track per-target correct counts and per-target loss
    correct_per_target = [0, 0, 0, 0, 0]
    loss_per_target = [0.0, 0.0, 0.0, 0.0, 0.0]
    
    # Element-wise loss calculator without reduction
    criterion_none = nn.BCEWithLogitsLoss(reduction='none')

    for x_batch, y_batch in dataloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        # Compute per-target loss breakdown: shape (batch_size, 5)
        elem_loss = criterion_none(logits, y_batch)
        elem_loss_sum = elem_loss.sum(dim=0).detach().cpu()
        for idx in range(5):
            loss_per_target[idx] += elem_loss_sum[idx].item()

        # Multi-label prediction
        probs = torch.sigmoid(logits)
        preds = (probs > threshold).float()

        # Exact match (all 5 targets must match)
        exact_matches = (preds == y_batch).all(dim=1)
        correct_exact += exact_matches.sum().item()

        # Total element-wise correct
        correct_elements += (preds == y_batch).sum().item()

        # Per-target element-wise correct
        for idx in range(5):
            correct_per_target[idx] += (preds[:, idx] == y_batch[:, idx]).sum().item()

        total_samples += y_batch.shape[0]
        total_elements += y_batch.numel()
        train_loss += loss.item() * y_batch.shape[0]

    avg_train_loss = train_loss / total_samples
    exact_acc = correct_exact / total_samples
    elements_acc = correct_elements / total_elements
    avg_loss_per_target = [l / total_samples for l in loss_per_target]

    return avg_train_loss, exact_acc, elements_acc, correct_per_target, avg_loss_per_target, total_samples


## Model Evaluation Logic

In [82]:
def eval_model(model, dataloader, criterion, device, threshold=0.5):
    model.to(device)
    model.eval()
    
    eval_loss = 0.0
    total_samples = 0
    total_elements = 0
    correct_exact = 0
    correct_elements = 0
    
    correct_per_target = [0, 0, 0, 0, 0]
    loss_per_target = [0.0, 0.0, 0.0, 0.0, 0.0]
    criterion_none = nn.BCEWithLogitsLoss(reduction='none')

    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            elem_loss = criterion_none(logits, y_batch)
            elem_loss_sum = elem_loss.sum(dim=0).cpu()
            for idx in range(5):
                loss_per_target[idx] += elem_loss_sum[idx].item()

            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()

            exact_matches = (preds == y_batch).all(dim=1)
            correct_exact += exact_matches.sum().item()
            correct_elements += (preds == y_batch).sum().item()

            for idx in range(5):
                correct_per_target[idx] += (preds[:, idx] == y_batch[:, idx]).sum().item()

            total_samples += y_batch.shape[0]
            total_elements += y_batch.numel()
            eval_loss += loss.item() * y_batch.shape[0]

    avg_eval_loss = eval_loss / total_samples
    exact_acc = correct_exact / total_samples
    elements_acc = correct_elements / total_elements
    avg_loss_per_target = [l / total_samples for l in loss_per_target]

    return avg_eval_loss, exact_acc, elements_acc, correct_per_target, avg_loss_per_target, total_samples


## Train & Evaluate Model

In [83]:
TARGET_NAMES = ['Target 1 (NORM)', 'Target 2 (MI)', 'Target 3 (STTC)', 'Target 4 (CD)', 'Target 5 (HYP)']

print("Training CNN Model....\n")
for epoch in range(1, EPOCHS + 1):
    (train_loss, exact_train_acc, elem_train_acc, 
     train_targets_acc, train_targets_loss, train_samples) = train_model(
        model, train_dataloader, criterion, optimizer, device, threshold=0.5
    )

    (val_loss, exact_val_acc, elem_val_acc, 
     val_targets_acc, val_targets_loss, val_samples) = eval_model(
        model, val_dataloader, criterion, device, threshold=0.5
    )

    print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Exact Train Acc: {exact_train_acc*100:.2f}% | Element Train Acc: {elem_train_acc*100:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Exact Val Acc: {exact_val_acc*100:.2f}% | Element Val Acc: {elem_val_acc*100:.2f}%")
    
    print("--- Per-Target Breakdown (Validation) ---")
    for idx, name in enumerate(TARGET_NAMES):
        acc_pct = (val_targets_acc[idx] / val_samples) * 100
        print(f"  {name:18s} | Loss: {val_targets_loss[idx]:.4f} | Acc: {val_targets_acc[idx]}/{val_samples} ({acc_pct:.2f}%)")
    print()

print("Training complete!")


Training CNN Model....



Epoch [1/20] | Train Loss: 0.4220 | Exact Train Acc: 45.50% | Element Train Acc: 81.13%
Val Loss: 0.3470 | Exact Val Acc: 53.53% | Element Val Acc: 84.15%
--- Per-Target Breakdown (Validation) ---
  Target 1 (NORM)    | Loss: 0.4219 | Acc: 1185/1500 (79.00%)
  Target 2 (MI)      | Loss: 0.2572 | Acc: 1356/1500 (90.40%)
  Target 3 (STTC)    | Loss: 0.3811 | Acc: 1231/1500 (82.07%)
  Target 4 (CD)      | Loss: 0.3373 | Acc: 1291/1500 (86.07%)
  Target 5 (HYP)     | Loss: 0.3377 | Acc: 1248/1500 (83.20%)

Epoch [2/20] | Train Loss: 0.3396 | Exact Train Acc: 55.65% | Element Train Acc: 85.37%
Val Loss: 0.3515 | Exact Val Acc: 57.73% | Element Val Acc: 84.56%
--- Per-Target Breakdown (Validation) ---
  Target 1 (NORM)    | Loss: 0.3149 | Acc: 1315/1500 (87.67%)
  Target 2 (MI)      | Loss: 0.2617 | Acc: 1350/1500 (90.00%)
  Target 3 (STTC)    | Loss: 0.3812 | Acc: 1240/1500 (82.67%)
  Target 4 (CD)      | Loss: 0.3809 | Acc: 1224/1500 (81.60%)
  Target 5 (HYP)     | Loss: 0.4189 | Acc: 1213

In [84]:
TARGET_NAMES = ['Target 1 (NORM)', 'Target 2 (MI)', 'Target 3 (STTC)', 'Target 4 (CD)', 'Target 5 (HYP)']

print("Evaluating the Model...")
val_loss, exact_acc, elem_acc, targets_acc, targets_loss, total_samples = eval_model(
    model, val_dataloader, criterion, device
)

print(f"Test Loss: {val_loss:.4f}")
print(f"Exact Match Accuracy: {exact_acc * 100:.2f}%")
print(f"Element-Wise Accuracy: {elem_acc * 100:.2f}%\n")

print("--- Per-Target Breakdown ---")
for idx, name in enumerate(TARGET_NAMES):
    acc_pct = (targets_acc[idx] / total_samples) * 100
    print(f"  {name:18s} | Loss: {targets_loss[idx]:.4f} | Acc: {targets_acc[idx]}/{total_samples} ({acc_pct:.2f}%)")

print("\nEvaluation complete")


Evaluating the Model...
Test Loss: 0.2517
Exact Match Accuracy: 67.73%
Element-Wise Accuracy: 89.51%

--- Per-Target Breakdown ---
  Target 1 (NORM)    | Loss: 0.2827 | Acc: 1327/1500 (88.47%)
  Target 2 (MI)      | Loss: 0.2040 | Acc: 1378/1500 (91.87%)
  Target 3 (STTC)    | Loss: 0.2040 | Acc: 1374/1500 (91.60%)
  Target 4 (CD)      | Loss: 0.2856 | Acc: 1306/1500 (87.07%)
  Target 5 (HYP)     | Loss: 0.2819 | Acc: 1328/1500 (88.53%)

Evaluation complete


In [ ]:
print("Evaluating the Model...")
loss, exact_acc, elem_acc, *_ = eval_model(model, val_dataloader, criterion, device)

print(f"Test Loss: {loss:.3f} | Exact Match Acc: {exact_acc * 100:.2f}% | Element-Wise Acc: {elem_acc * 100:.2f}%")
print("Evaluation complete")

Evaluating the Model...
Test Loss: 0.252 | Exact Match Acc: 67.73% | Element-Wise Acc: 89.51%
Evaluation complete
